# Sesión 8 · App terminada y demo

**Módulo 4 · Tu primera app de IA** · 2 sesiones · proyecto real · de idea a producto

> Última sesión del curso. Convertimos el prototipo en una app cuidada, la publicamos en internet para que cualquiera la use y la presentamos. Cerramos con un mapa de hacia dónde seguir.

**Al terminar la sesión sabrás:**
- Construir una interfaz de **chat con memoria**
- Reforzar la lógica y el manejo de errores
- **Desplegar** la app en internet (Streamlit Community Cloud)
- Presentar una demo y saber qué aprender después

*Resultado de la sesión: tienes una app de IA funcional, publicada y en tu portfolio.*

## 1. De prototipo a producto: ¿qué le falta?

El prototipo de la Sesión 7 funciona, pero como producto se queda corto. Repasemos qué mejorar:

- **Memoria** — ahora cada pregunta es independiente; un buen chat **recuerda** la conversación
- **Interfaz de chat** — una caja y un botón saben a poco; queremos burbujas de chat de verdad
- **Elección de proveedor** — dar a elegir entre Claude, OpenRouter y Ollama
- **Robustez** — que no se rompa si falla la red o falta la clave (manejo de errores)
- **Claves seguras** — no escribir la clave en el código, sobre todo de cara a publicarla

Vamos pieza por pieza y luego lo unimos todo.

## 2. Interfaz de chat con memoria

Streamlit trae tres herramientas que nos resuelven el chat casi enteras:

- **`st.chat_message("user" / "assistant")`** — dibuja la burbuja de cada mensaje
- **`st.chat_input(...)`** — la caja de escritura fija abajo, como en cualquier mensajería
- **`st.session_state`** — la "memoria" de la app entre interacciones. Aquí guardamos la lista de mensajes

La idea de la memoria es la misma de la Sesión 6: mantenemos una **lista de mensajes** y, en cada turno, la mostramos entera y se la enviamos completa al modelo. Streamlit reejecuta el script en cada interacción, por eso necesitamos `session_state` para no perder el historial.

## 3. Reforzar la lógica

Tres detalles que convierten el prototipo en algo serio:

- **Manejo de errores** — envolvemos la llamada al LLM en `try/except`. Si algo falla, mostramos un mensaje amable en vez de una pantalla de error
- **Caché del PDF** — con `@st.cache_data` leemos el PDF una sola vez, no en cada mensaje (más rápido)
- **System prompt afinado** — insistimos en que responda solo desde las FAQ y reconozca cuando no sabe algo, para evitar invenciones

## 4. La app final, completa

Esta celda guarda la versión definitiva en `chatbot_faq.py`. Reúne todo lo anterior: lectura del PDF en caché, selección de proveedor (los tres del curso), claves seguras, interfaz de chat con memoria y manejo de errores. Léela con calma: reconocerás cada bloque de lo que hemos ido haciendo.

In [ ]:
%%writefile chatbot_faq.py
# -*- coding: utf-8 -*-
"""
Chatbot de Preguntas Frecuentes (FAQ) para Atención al Cliente.
Proyecto final del curso · Módulo 4.

Cómo ejecutarlo:
    1) pip install -r requirements.txt
    2) streamlit run chatbot_faq.py

La app lee un PDF de FAQ, lo entrega como contexto a un LLM y responde
las dudas del cliente basándose SOLO en esa información.

Soporta tres proveedores: Claude (Anthropic), OpenRouter (gratis) y Ollama (local).
"""
import os
import streamlit as st
from pypdf import PdfReader

# --------------------------------------------------------------------------
# 1. CONFIGURACIÓN BÁSICA DE LA PÁGINA
# --------------------------------------------------------------------------
st.set_page_config(page_title="Asistente de FAQ", page_icon="💬")
RUTA_FAQ = "faq_horta_verde.pdf"   # cambia esto por tu propio PDF


# --------------------------------------------------------------------------
# 2. LEER EL PDF DE FAQ (en caché para no releerlo en cada interacción)
# --------------------------------------------------------------------------
@st.cache_data
def cargar_faq(ruta):
    """Lee el PDF y devuelve todo su texto como una cadena."""
    lector = PdfReader(ruta)
    texto = ""
    for pagina in lector.pages:
        texto += pagina.extract_text() + "\n"
    return texto


# --------------------------------------------------------------------------
# 3. LLAMAR AL LLM (tres proveedores posibles)
# --------------------------------------------------------------------------
def responder(historial, system, proveedor, clave):
    """Envía el system prompt + el historial al proveedor elegido y devuelve el texto."""
    if proveedor == "Claude (Anthropic)":
        import anthropic
        cliente = anthropic.Anthropic(api_key=clave)
        r = cliente.messages.create(
            model="claude-haiku-4-5",
            max_tokens=500,
            system=system,
            messages=historial,
        )
        return r.content[0].text

    elif proveedor == "OpenRouter (gratis)":
        from openai import OpenAI
        cliente = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=clave)
        mensajes = [{"role": "system", "content": system}] + historial
        r = cliente.chat.completions.create(
            model="meta-llama/llama-4-maverick:free",
            messages=mensajes,
            max_tokens=500,
        )
        return r.choices[0].message.content

    else:  # Ollama (local)
        from ollama import chat
        mensajes = [{"role": "system", "content": system}] + historial
        return chat(model="llama3.2", messages=mensajes).message.content


# --------------------------------------------------------------------------
# 4. BARRA LATERAL: elegir proveedor e introducir la clave
# --------------------------------------------------------------------------
st.sidebar.header("Configuración")
proveedor = st.sidebar.selectbox(
    "Proveedor del modelo",
    ["Claude (Anthropic)", "OpenRouter (gratis)", "Ollama (local)"],
)

def leer_secreto(nombre):
    """Lee una clave de st.secrets sin fallar si no existe el archivo de secretos."""
    try:
        return st.secrets[nombre]
    except Exception:
        return ""

# La clave se busca primero en los 'secrets' (para el despliegue), luego en las
# variables de entorno y, si no, se pide en la barra lateral
clave = ""
if proveedor != "Ollama (local)":
    nombre_secreto = "ANTHROPIC_API_KEY" if proveedor.startswith("Claude") else "OPENROUTER_API_KEY"
    clave = leer_secreto(nombre_secreto) or os.environ.get(nombre_secreto, "")
    if not clave:
        clave = st.sidebar.text_input("Clave API", type="password")

if st.sidebar.button("🗑️ Reiniciar conversación"):
    st.session_state.mensajes = []
    st.rerun()


# --------------------------------------------------------------------------
# 5. INSTRUCCIONES DEL ASISTENTE (system prompt)
# --------------------------------------------------------------------------
texto_faq = cargar_faq(RUTA_FAQ)

SYSTEM = f"""Eres el asistente virtual de atención al cliente de Horta Verde, una tienda online.
Tu tarea es responder las dudas de los clientes de forma amable, clara y breve, en español.

Reglas importantes:
- Responde ÚNICAMENTE con la información de las Preguntas Frecuentes que aparecen más abajo.
- Si la respuesta no está en las FAQ, dilo con honestidad e invita a contactar con atención
  al cliente (hola@hortaverde.example o 900 123 456). No te inventes datos.
- Sé cercano y profesional. No uses tecnicismos.

=== PREGUNTAS FRECUENTES ===
{texto_faq}
=== FIN DE LAS FAQ ==="""


# --------------------------------------------------------------------------
# 6. INTERFAZ DE CHAT
# --------------------------------------------------------------------------
st.title("💬 Asistente de atención al cliente")
st.caption("Pregúntame sobre pedidos, envíos, devoluciones, pagos o tu cuenta.")

# Inicializar el historial de la conversación
if "mensajes" not in st.session_state:
    st.session_state.mensajes = []

# Mostrar los mensajes anteriores
for m in st.session_state.mensajes:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

# Caja de entrada del usuario
pregunta = st.chat_input("Escribe tu pregunta...")
if pregunta:
    # Guardar y mostrar la pregunta del usuario
    st.session_state.mensajes.append({"role": "user", "content": pregunta})
    with st.chat_message("user"):
        st.markdown(pregunta)

    # Validar que hay clave si el proveedor la necesita
    if proveedor != "Ollama (local)" and not clave:
        with st.chat_message("assistant"):
            st.warning("Introduce tu clave API en la barra lateral para poder responder.")
    else:
        with st.chat_message("assistant"):
            with st.spinner("Pensando..."):
                try:
                    respuesta = responder(st.session_state.mensajes, SYSTEM, proveedor, clave)
                except Exception as error:
                    respuesta = (f"Lo siento, ha ocurrido un problema técnico "
                                 f"({type(error).__name__}). Inténtalo de nuevo en unos segundos.")
            st.markdown(respuesta)
        st.session_state.mensajes.append({"role": "assistant", "content": respuesta})


Lánzala en tu terminal:

```bash
pip install -r requirements.txt
streamlit run chatbot_faq.py
```

Elige el proveedor en la barra lateral, introduce tu clave (o usa Ollama sin clave) y conversa. Prueba a preguntar algo, y luego una pregunta de seguimiento que dependa de la anterior, para comprobar que **recuerda**.

> El archivo `requirements.txt` (te lo paso con el material) lista las librerías que necesita la app. Es imprescindible para el despliegue del siguiente paso.

## 5. Desplegar: que tu app viva en internet

Hasta ahora la app solo corre en tu equipo. **Desplegar** = ponerla en un servidor para que cualquiera la abra con un enlace. La vía más sencilla y gratuita es **Streamlit Community Cloud**.

Los pasos, en orden:

1. **Sube tu proyecto a GitHub** — crea un repositorio gratuito con tres archivos: `chatbot_faq.py`, `requirements.txt` y `faq_horta_verde.pdf`
2. **Entra en `share.streamlit.io`** con tu cuenta de GitHub
3. **Pulsa "New app"** y selecciona tu repositorio y el archivo `chatbot_faq.py`
4. **Configura la clave de forma segura** — en *Advanced settings → Secrets*, pega tu clave así:
   ```toml
   ANTHROPIC_API_KEY = "sk-ant-..."
   ```
   La app la lee con `st.secrets`, sin que la clave quede nunca en el código público
5. **Pulsa "Deploy"** — en uno o dos minutos tendrás un enlace público para compartir

> ⚠️ **Nunca subas tu clave a GitHub.** Por eso usamos *Secrets*: el código va público, la clave no. Si una clave se filtra, anúlala de inmediato en la consola del proveedor.

## 6. Demo y revisión del proyecto

Antes de presentar, pasa esta **lista de comprobación** (pruébalo todo en vivo):

- Responde bien a preguntas que **están** en las FAQ (envíos, devoluciones, pagos)
- Ante una pregunta que **no está**, reconoce que no lo sabe y deriva a soporte (no inventa)
- **Recuerda** el contexto en una pregunta de seguimiento
- No se rompe si te equivocas de clave o se va la conexión (muestra un mensaje amable)
- El botón de **reiniciar** limpia la conversación

**Guion sugerido para la demo (3 minutos):**
1. Explica el problema en una frase: "atención al cliente 24/7 sin inventar nada"
2. Muestra la arquitectura: PDF → LLM → web
3. Haz dos preguntas en vivo: una sencilla y una de seguimiento
4. Provoca a propósito una pregunta fuera de las FAQ para enseñar que **no alucina**
5. Cierra con el enlace público desplegado

## 7. Próximos pasos: qué aprender después

Has cerrado el ciclo completo: de no saber qué es la IA a publicar una app. Si quieres seguir creciendo, estas son las direcciones naturales:

- **RAG (búsqueda + generación)** — para documentos grandes que no caben en el contexto. Se trocea el texto, se convierte en vectores (*embeddings*) y se recupera solo lo relevante para cada pregunta. Es el siguiente nivel de este mismo proyecto
- **Evaluación** — medir de forma sistemática si las respuestas son correctas, con un conjunto de preguntas de prueba
- **Más fuentes de datos** — conectar varios PDF, una web o una base de datos
- **Registro y mejora continua** — guardar las preguntas reales para detectar lagunas en las FAQ
- **Otras interfaces** — integrarlo en WhatsApp, en la web de la empresa o en un widget

> Mi recomendación: **no acumules teoría, construye**. Coge un problema real de tu entorno y repite este ciclo —datos, modelo, interfaz— con él. Se aprende enviando proyectos, no leyendo sobre ellos.

## 8. Resumen del curso

Mira atrás un momento. En ocho sesiones has recorrido:

- **Módulo 1** — qué es la IA y cómo aprenden las máquinas, sin una sola fórmula
- **Módulo 2** — Python desde cero y tu primer modelo con scikit-learn
- **Módulo 3** — los grandes modelos de lenguaje, *prompt engineering* y sus APIs
- **Módulo 4** — un proyecto real de principio a fin: un chatbot de FAQ publicado

Pasaste de **cero** a tener una app de IA funcionando en internet. Ese salto es exactamente lo que distingue a quien "ha oído hablar de IA" de quien **sabe aplicarla**.

👉 **Publica tu proyecto y compártelo.** Es la mejor carta de presentación de lo que ya sabes hacer.

¡Enhorabuena por completar el curso!